<a href="https://colab.research.google.com/github/nndna/Neurodata-Machine-Learning---HW1/blob/main/HW1_merged_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning & Neural Networks for Neuroscience — HW1

**Combined notebook (Parts 1–3)**

We use one recording session and mainly study **V1 neuron 61**. We also look at neuron 0 as an extra example in Part 3.

The code is kept in the same order as the assignment, so we can run the notebook from top to bottom.

## 0. Load our data

Upload `106r002p70_spikes.mat` to Colab (Files panel). This is the session we used in both notebooks. The file contains trial information and spike recordings for V1 and V2.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

# Use the same simple plot style throughout the notebook
plt.rcParams.update({'figure.figsize': (8, 5), 'font.size': 11})

file_path = '/content/106r002p70_spikes.mat'

# If we run the notebook outside Colab, look for the file in this folder
if not Path(file_path).exists():
    file_path = '106r002p70_spikes.mat'

with h5py.File(file_path, 'r') as f:
    stim_data = np.array(f['stim'])
    spike_refs = f['spikes']
    v1_spikes = np.array(f[spike_refs[0, 0]])
    v2_spikes = np.array(f[spike_refs[1, 0]])

print('V1:', v1_spikes.shape)
print('V2:', v2_spikes.shape)
print('Stimuli:', stim_data.shape)

## Part 1. From raw spikes to a tuning curve

Each trial lasts **1280 ms**. First, we add up the spikes over time. Then, we group the trials using the stimulus code and calculate the average response of every neuron.

In [ ]:
# The second axis is time, so we add the 1280 time bins
spike_counts = np.sum(v1_spikes, axis=1)

# Get one stimulus code per trial
stim = stim_data.flatten()
orientations, trial_counts = np.unique(stim, return_counts=True)

print('Spike count shape:', spike_counts.shape)
print('Stimulus codes:', orientations)
print('Trials for each code:', trial_counts)
print('Number of V1 neurons:', v1_spikes.shape[2])

In [ ]:
# One row for each neuron and one column for each orientation
mean_responses = np.zeros((v1_spikes.shape[2], len(orientations)))

for neuron in range(v1_spikes.shape[2]):
    for i, ori in enumerate(orientations):
        # Only take trials that used this orientation
        trials = spike_counts[stim == ori, neuron]
        mean_responses[neuron, i] = np.mean(trials)

print('Mean responses:', mean_responses.shape)

The stimulus codes go from 1 to 8. The dataset documentation describes eight orientations separated by 22.5°. For now, our plots use **stimulus codes** to match our earlier regression work. We will use degrees when circularity matters.

In [ ]:
# Look at a few neurons, including our selected neuron 61
selected_neurons = [60, 61, 62, 63, 64, 65]

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)

for ax, neuron in zip(axes.flat, selected_neurons):
    ax.plot(orientations, mean_responses[neuron], 'o-', color='tab:blue')
    ax.set_title(f'Neuron {neuron}')
    ax.set_xticks(orientations)
    ax.grid(alpha=0.25)

for ax in axes[-1]:
    ax.set_xlabel('Stimulus orientation (code)')
for ax in axes[:, 0]:
    ax.set_ylabel('Mean spike count per trial')

fig.suptitle('Orientation tuning curves — V1')
plt.tight_layout()
plt.show()

**Our choice:** We use **neuron 61** for the next parts because its average response has a fairly clear peak near code 5 and lower values towards the ends. This looks reasonably bell-shaped.

## Part 2. Polynomial regression

From now on, we use **spikes per second** (the TA's requested units). A trial is 1.28 seconds long, so we divide spike counts by 1.28. We fit polynomial models of degrees 1, 3, 4 and 7 to the eight mean responses of neuron 61.

In [ ]:
# Convert the mean response to spikes per second
mean_rates = mean_responses / 1.28

neuron_idx = 61
x = orientations
y = mean_rates[neuron_idx]

In [ ]:
# Fit each polynomial and calculate its MSE
polynomial_degrees = [1, 3, 4, 7]
polynomial_fits = {}
polynomial_mse = {}

for degree in polynomial_degrees:
    coefficients = np.polyfit(x, y, degree)
    predictions = np.polyval(coefficients, x)
    mse = np.mean((y - predictions) ** 2)
    polynomial_fits[degree] = coefficients
    polynomial_mse[degree] = mse
    print(f'Degree {degree} MSE: {mse:.4f}')

In [ ]:
# Plot all four models in the same style
x_smooth = np.linspace(x.min(), x.max(), 200)
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)

for ax, degree in zip(axes.flat, polynomial_degrees):
    y_smooth = np.polyval(polynomial_fits[degree], x_smooth)
    ax.scatter(x, y, color='black', label='Experimental data', zorder=3)
    ax.plot(x_smooth, y_smooth, color='tab:blue', label='Polynomial fit')
    ax.set_title(f'Degree {degree} (MSE = {polynomial_mse[degree]:.4f})')
    ax.set_xticks(x)
    ax.grid(alpha=0.25)

for ax in axes[-1]:
    ax.set_xlabel('Stimulus orientation (code)')
for ax in axes[:, 0]:
    ax.set_ylabel('Mean firing rate (spikes/s)')

axes[0, 0].legend()
fig.suptitle('Polynomial regression — V1 neuron 61')
plt.tight_layout()
plt.show()

A degree-7 polynomial can pass through all eight mean response points. This gives a very low training MSE, but it does not tell us whether the model would work well on new trials.

## Part 3. Nonlinear regression — Gaussian

Here we fit a Gaussian curve. Its parameters describe the baseline response ($B$), height above baseline ($A$), preferred orientation ($\mu$), and width ($\sigma$).

In [ ]:
def gaussian(theta, B, A, mu, sigma):
    return B + A * np.exp(-((theta - mu) ** 2) / (2 * sigma ** 2))

# Start with values suggested by the data
B0 = np.min(y)
A0 = np.max(y) - B0
mu0 = x[np.argmax(y)]
sigma0 = 1.0

params, covariance = curve_fit(
    gaussian, x, y,
    p0=[B0, A0, mu0, sigma0],
    bounds=([0, 0, 1, 0.01], [np.inf, np.inf, 8, np.inf])
)

B, A, mu, sigma = params
y_pred_gaussian = gaussian(x, B, A, mu, sigma)
mse_gaussian = np.mean((y - y_pred_gaussian) ** 2)

print(f'Baseline (B): {B:.3f} spikes/s')
print(f'Amplitude (A): {A:.3f} spikes/s')
print(f'Preferred orientation (mu): {mu:.3f} codes')
print(f'Tuning width (sigma): {sigma:.3f} code units')
print(f'Gaussian MSE: {mse_gaussian:.4f}')

In [ ]:
# Show the data and our Gaussian curve
plt.figure()
plt.scatter(x, y, color='black', label='Experimental data', zorder=3)
plt.plot(x_smooth, gaussian(x_smooth, B, A, mu, sigma),
         color='tab:blue', label='Gaussian fit')
plt.title('Gaussian tuning curve — V1 neuron 61')
plt.xlabel('Stimulus orientation (code)')
plt.ylabel('Mean firing rate (spikes/s)')
plt.xticks(x)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

### Extra example: neuron 0

We also tried a neuron that does not have the same clear bell shape. This is just an extra comparison; **neuron 61 is still our main neuron**. We use the same Gaussian function and settings.

In [ ]:
# Repeat the Gaussian fit for neuron 0
other_neuron = 0
y0 = mean_rates[other_neuron]

B0 = np.min(y0)
A0 = np.max(y0) - B0
mu0 = x[np.argmax(y0)]
sigma0 = 1.0

params0, covariance0 = curve_fit(
    gaussian, x, y0,
    p0=[B0, A0, mu0, sigma0],
    bounds=([0, 0, 1, 0.01], [np.inf, np.inf, 8, np.inf])
)

B_0, A_0, mu_0, sigma_0 = params0
mse_gaussian0 = np.mean((y0 - gaussian(x, *params0)) ** 2)

print(f'Baseline (B): {B_0:.3f} spikes/s')
print(f'Amplitude (A): {A_0:.3f} spikes/s')
print(f'Preferred orientation (mu): {mu_0:.3f} codes')
print(f'Tuning width (sigma): {sigma_0:.3f} code units')
print(f'Gaussian MSE: {mse_gaussian0:.4f}')

In [ ]:
# Same axes and colors for both Gaussian fits
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

for ax, neuron, responses, fit_params in [
    (axes[0], 61, y, params),
    (axes[1], 0, y0, params0)
]:
    ax.scatter(x, responses, color='black', label='Experimental data', zorder=3)
    ax.plot(x_smooth, gaussian(x_smooth, *fit_params),
            color='tab:blue', label='Gaussian fit')
    ax.set_title(f'V1 neuron {neuron}')
    ax.set_xlabel('Stimulus orientation (code)')
    ax.set_xticks(x)
    ax.grid(alpha=0.25)

axes[0].set_ylabel('Mean firing rate (spikes/s)')
axes[1].set_ylabel('Mean firing rate (spikes/s)')
axes[0].legend()
fig.suptitle('Gaussian tuning: two different neurons')
plt.tight_layout()
plt.show()

In [ ]:
# Compare the model parameters
print('Parameter             Neuron 61    Neuron 0')
print(f'Baseline (spikes/s)   {B:9.3f}   {B_0:9.3f}')
print(f'Amplitude (spikes/s)  {A:9.3f}   {A_0:9.3f}')
print(f'Mu (code)             {mu:9.3f}   {mu_0:9.3f}')
print(f'Sigma (code units)    {sigma:9.3f}   {sigma_0:9.3f}')
print(f'MSE                    {mse_gaussian:8.4f}    {mse_gaussian0:8.4f}')

Neuron 0 may appear to have a very narrow Gaussian peak, but that peak is between the measured orientations. We should therefore be careful when interpreting its small $\sigma$ as strong orientation selectivity.

### MSE comparison for the main neuron

In [ ]:
# Compare the models that we have fitted to neuron 61
for degree in polynomial_degrees:
    print(f'Polynomial degree {degree}: {polynomial_mse[degree]:.4f}')
print(f'Gaussian: {mse_gaussian:.4f}')

## Part 4. Circular orientation — next step

Orientation is circular: 0° and 180° describe the same orientation. We have not fitted the von Mises model in the two original notebooks yet. Here is the function from our first notebook, ready for the next part. **We should not use the codes 1–8 as degrees.**

In [ ]:
def von_mises(theta_deg, B, A, kappa, mu_deg):
    # Convert degrees to radians because np.cos uses radians
    theta_rad = np.deg2rad(theta_deg)
    mu_rad = np.deg2rad(mu_deg)
    return B + A * np.exp(kappa * np.cos(2 * (theta_rad - mu_rad)))

# For the circular model, the eight angles are:
angles_deg = (orientations - 1) * 22.5
print('Angles in degrees:', angles_deg)

# We will fit this model in Part 4.

---
**Notes:** This notebook combines the code in the two uploaded files. Old trial-and-error cells and repeated imports were removed. We need the original `.mat` file to rerun it; numerical output is calculated when the file is loaded. The von Mises function is included but **not fitted yet**.